<a href="https://colab.research.google.com/github/victoriashushpannikova/nlp_homeworks/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22w2v_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [ ]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [ ]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)


1. word2vec-google-news-300

Источник данных: модель обучена на части датасета Google News — новостных статей на английском языке, содержащей около 100 миллиардов слов.

Примерный объём: модель включает 300-мерные векторы для 3 миллионов слов и фраз.

Для чего используется:
  1) Может быть использована для поиска семантически близких слов, вычисления схожести документов и кластеризации текстов.
  2) Может решать аналогии, например, по формуле "king" — "man" + "woman" ≈ "queen".

Применение: используется в системах рекомендаций, для извлечения признаков при анализе тональности, а также в агрегаторах новостей.

2. fasttext-wiki-news-subwords-300

Источник данных: модель обучена на комбинации из трёх крупных корпусов: Wikipedia 2017 года, веб-корпус UMBC и новостной датасет statmt.org.

Примерный объём: Размер векторов — 300, а модель содержит 1 миллион слов.

Для чего используется:
1) Работа с редкими словами и опечатками: представляет каждое слово как набор n-грамм. Благодаря этому модель может вычислить вектор даже для слова, которого не было в обучающем корпусе, или для слова с опечаткой. Также, некоторые слова, отсутствующие в модели word2vec-google-news-300, успешно находятся в fasttext-wiki-news-subwords-300. Она также способна улавливать смысл на уровне морфем.

3. glove-twitter-25

Источник данных: Модель обучена из корпуса из 2 миллиардов твитов.

Примерный объём: Имеет небольшую размерность — 25, словарь содержит 1.2 миллиона слов.

Для чего используется:
1) Анализ социальных сетей: обработка неформального короткого текста. Модель понимает сленг, хештеги, современные идиомы и эмоционально окрашенные выражения.
2) Сентимент-анализ (анализ тональности): Благодаря неформальному контексту, она лучше справляется с определением эмоций и сарказма в коротких сообщениях.
3) Подходит для проектов с ограниченными ресурсами, где не нужна глубокая семантическая сложность, но важна обработка современного языка интернета.

**Базовые операции с векторами**

In [ ]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [ ]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [3]:
!pip install gensim
import gensim.downloader as api
w2v_google_model = api.load('word2vec-google-news-300')

print(f"Размер словаря: {len(w2v_google_model.key_to_index)}")
print(f"Размерность векторов: {w2v_google_model.vector_size}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.7 MB/s eta 0:00:00
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Размер словаря: 3000000
Размерность векторов: 300


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [5]:
def find_similar_words(word, model=w2v_google_model, topn=10):
    """Находит топ-N наиболее похожих слов для заданного слова в модели."""
    try:
        similar = model.most_similar(word, topn=topn)
        print(f"Слова, похожие на '{word}':")
        for w, score in similar:
            print(f"  {w}: {score:.4f}")
    except KeyError:
        print(f"Слово '{word}' не найдено в словаре модели.")

# Пример использования функции
find_similar_words('python')
find_similar_words('dog')
find_similar_words('car')

Слова, похожие на 'python':
  pythons: 0.6688
  Burmese_python: 0.6680
  snake: 0.6606
  crocodile: 0.6591
  boa_constrictor: 0.6444
  alligator: 0.6422
  reptile: 0.6388
  albino_python: 0.6159
  croc: 0.6084
  lizard: 0.6013
Слова, похожие на 'dog':
  dogs: 0.8680
  puppy: 0.8106
  pit_bull: 0.7804
  pooch: 0.7627
  cat: 0.7609
  golden_retriever: 0.7501
  German_shepherd: 0.7465
  Rottweiler: 0.7438
  beagle: 0.7419
  pup: 0.7407
Слова, похожие на 'car':
  vehicle: 0.7821
  cars: 0.7424
  SUV: 0.7161
  minivan: 0.6907
  truck: 0.6736
  Car: 0.6678
  Ford_Focus: 0.6673
  Honda_Civic: 0.6627
  Jeep: 0.6511
  pickup_truck: 0.6441


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [9]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [10]:
from gensim.models import Word2Vec
# Обучаем модель Word2Vec на тестовом датасете
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1  # Использовать skip-gram
)

model.train(cooking_sentences, total_examples=model.corpus_count, epochs=model.epochs)
print("Модель Word2Vec обучена!")

Модель Word2Vec обучена!


In [11]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [12]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2397
  ингредиенты: 0.2173
  хлеб: 0.1941
  брокколи: 0.1846
  кипятить: 0.1714


In [13]:
# Найдите слова, похожие на "духовка"
try:
    similar_oven = model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar_oven:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

# Найдите слова, похожие на "овощи"
try:
    similar_veg = model.wv.most_similar('овощи', topn=5)
    print("\nСлова, похожие на 'овощи':")
    for word, score in similar_veg:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'духовка':
  ингредиенты: 0.3189
  десерт: 0.3071
  холодильник: 0.2696
  питание: 0.2242
  пирог: 0.2150

Слова, похожие на 'овощи':
  мариновать: 0.2724
  хлеб: 0.2691
  гриль: 0.2556
  фольга: 0.2428
  сахар: 0.2120


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [15]:
from gensim.models import FastText
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [16]:
# Найдите слова, похожие на "варить" с помощью FastText
try:
    similar_ft_varit = ft_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить' (FastText):")
    for word, score in similar_ft_varit:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре FastText")

# Найдите слова, похожие на "духовка" с помощью FastText
try:
    similar_ft_duhovka = ft_model.wv.most_similar('духовка', topn=5)
    print("\nСлова, похожие на 'духовка' (FastText):")
    for word, score in similar_ft_duhovka:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре FastText")

# Найдите слова, похожие на "овощи" с помощью FastText
try:
    similar_ft_ovoschi = ft_model.wv.most_similar('овощи', topn=5)
    print("\nСлова, похожие на 'овощи' (FastText):")
    for word, score in similar_ft_ovoschi:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре FastText")

Слова, похожие на 'варить' (FastText):
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622

Слова, похожие на 'духовка' (FastText):
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944

Слова, похожие на 'овощи' (FastText):
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [17]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для слов с опечатками из кулинарной тематики
compare_models('вартить') # Опечатка в 'варить'
compare_models('духофка') # Опечатка в 'духовка'
compare_models('овощии') # Опечатка в 'овощи'


Сравнение для слова: 'вартить'
  Word2Vec: слово не найдено
  FastText: ['кипятить', 'варить']

Сравнение для слова: 'духофка'
  Word2Vec: слово не найдено
  FastText: ['бекон', 'травы']

Сравнение для слова: 'овощии'
  Word2Vec: слово не найдено
  FastText: ['овощи', 'сахар']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [27]:
from gensim.models.doc2vec import TaggedDocument

# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [28]:
from gensim.models import Doc2Vec

# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [29]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [30]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [31]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(2, 4) # python programming for data science vs computer vision processes images

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [32]:
# Найдите самый похожий документ на doc_1
similar_to_doc1 = doc_model.dv.most_similar("doc_1", topn=1)

print(f"Самый похожий документ на doc_1:")
for doc_tag, similarity in similar_to_doc1:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Самый похожий документ на doc_1:
  doc_0: 0.2735
    Текст: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [33]:
# 10. Обучите модели с разной размерностью и продемонстрируйте качество

# Выбираем Word2Vec и корпус cooking_sentences

def train_and_evaluate_word2vec(vector_size, sentences, examples_count, epochs):
    print(f"\nОбучение Word2Vec с vector_size={vector_size}...")
    model_config = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=3,
        min_count=1,
        workers=2,
        sg=1
    )
    model_config.train(sentences, total_examples=examples_count, epochs=epochs)
    print(f"Word2Vec (vector_size={vector_size}) обучена!")
    return model_config

# Обучаем модели с разной размерностью
model_w2v_10 = train_and_evaluate_word2vec(10, cooking_sentences, len(cooking_sentences), 10)
model_w2v_50 = train_and_evaluate_word2vec(50, cooking_sentences, len(cooking_sentences), 10)
model_w2v_100 = train_and_evaluate_word2vec(100, cooking_sentences, len(cooking_sentences), 10)


# Функция для демонстрации похожих слов
def demonstrate_similar_words(model, model_name, words_to_check):
    print(f"\n--- Демонстрация для {model_name} ---")
    for word in words_to_check:
        try:
            similar = model.wv.most_similar(word, topn=3)
            print(f"  Слова, похожие на '{word}':")
            for w, score in similar:
                print(f"    {w}: {score:.4f}")
        except KeyError:
            print(f"  Слово '{word}' не найдено в словаре {model_name}.")

# Примеры слов из кулинарной тематики для проверки
example_words = ['варить', 'духовка', 'мясо']

# Демонстрация качества для каждой модели
demonstrate_similar_words(model_w2v_10, "Word2Vec (vector_size=10)", example_words)
demonstrate_similar_words(model_w2v_50, "Word2Vec (vector_size=50)", example_words)
demonstrate_similar_words(model_w2v_100, "Word2Vec (vector_size=100)", example_words)


Обучение Word2Vec с vector_size=10...
Word2Vec (vector_size=10) обучена!

Обучение Word2Vec с vector_size=50...
Word2Vec (vector_size=50) обучена!

Обучение Word2Vec с vector_size=100...
Word2Vec (vector_size=100) обучена!

--- Демонстрация для Word2Vec (vector_size=10) ---
  Слова, похожие на 'варить':
    рыба: 0.6663
    сковорода: 0.6223
    хлеб: 0.6129
  Слова, похожие на 'духовка':
    взбивать: 0.5895
    тушить: 0.5875
    говядина: 0.5467
  Слова, похожие на 'мясо':
    жарить: 0.7010
    курица: 0.6488
    мука: 0.5818

--- Демонстрация для Word2Vec (vector_size=50) ---
  Слова, похожие на 'варить':
    вино: 0.2396
    ингредиенты: 0.2175
    хлеб: 0.1940
  Слова, похожие на 'духовка':
    ингредиенты: 0.3178
    десерт: 0.3076
    холодильник: 0.2693
  Слова, похожие на 'мясо':
    завтрак: 0.3768
    жарить: 0.2312
    взбивать: 0.2294

--- Демонстрация для Word2Vec (vector_size=100) ---
  Слова, похожие на 'варить':
    чашка: 0.3194
    вино: 0.2043
    сковорода: 0.19